# NeuroGuard — Phase 4: matched Tversky U-Net
This project is a prototype and is not medical advice.

Run in **browser Colab with a T4 GPU**, at the reviewed full source commit.
Phase 3 BCE is complete on user-verified validation evidence. This notebook
trains only Tversky; the original split, BCE model and BCE results stay read-only.
Keep runtime outputs private. No final test predictions, new split or Gradio.

Only loss and experiment identity change. Frozen Tversky: alpha=.3 FP, beta=.7 FN,
smoothing=1e-6 in numerator/denominator, per-slice then batch mean including empties.
Empty-mask FP gradients can be extremely small; review this limitation and the
train-only smoke before the definitive run. Do not tune the loss after results.
After reset rerun 1–10 at the SAME SHA/config/run ID, then RESUME in 11.
Completed epochs count toward the original 30-epoch budget. Real GPU execution
is performed by the student; no Tversky validation numbers are pre-filled.


## 1. Verify Colab and GPU
Select Runtime → Change runtime type → T4 GPU. Preserve the installed CUDA PyTorch.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from google.colab import drive
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing')
print('GPU:', torch.cuda.get_device_name(0))
if 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Use the same T4 hardware as the BCE experiment')


## 2. Enter the reviewed full GitHub commit SHA
After an approved push, open this notebook at that exact commit. Opening a notebook does not clone src/.


In [ ]:
REPOSITORY = 'https://github.com/hrb31/NeuroGuard.git'
REVISION = input('Full reviewed Phase 4 Git commit SHA: ').strip().lower()
if not re.fullmatch(r'[0-9a-f]{40}', REVISION):
    raise ValueError('Full 40-character Git SHA required')
REPO = Path('/content/NeuroGuard')


## 3. Clone the public repository and verify the revision
No private GitHub token is required. An edited or different checkout stops;
save outputs and use a clean runtime. Source is available only after an approved push.


In [ ]:
env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPOSITORY, str(REPO)], env=env, check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', REVISION], env=env, check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual != REVISION:
    raise RuntimeError('Existing checkout differs; save outputs and use a clean runtime')
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout has local changes; use a clean runtime')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Executing source commit:', actual)


## 4. Supporting dependencies, automated tests and environment
Keep CUDA PyTorch. Requirements include `jedi` for the observed Colab/IPython
missing-dependency issue. `pip check` remains strict: return genuine conflicts.
If an imported supporting package changes, restart and rerun at the same SHA.


In [ ]:
torch_before = (str(torch.__version__), torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-training.txt'], check=True)
torch_after = json.loads(subprocess.check_output([sys.executable, '-c',
    'import torch,json; print(json.dumps([str(torch.__version__), torch.version.cuda]))'], text=True))
assert tuple(torch_after) == torch_before, 'PyTorch installation changed: stop'
assert torch.cuda.is_available()
test_run = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test_run.stdout)
test_run.check_returncode()
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


## 5. Mount Drive and choose a separate Tversky run
Example run ID: `tversky-seed42-01`. Choose NEW, or the same ID and RESUME after
interruption. Never point this run at Phase 3. Preflight evidence is timestamped.
Read the BCE identity to check matched settings/hardware; never load its weights.
Core runtime differences stop. Supporting-package differences are saved for review.


In [ ]:
drive.mount('/content/drive')
from datetime import datetime, timezone
from src.data import write_json, digest
from src.training import seed_everything
PROJECT = Path('/content/drive/MyDrive/NeuroGuard')
PHASE2 = PROJECT / 'dataset-v2-seed42'
RUN_ID = input('Run ID (letters, digits, hyphen, underscore): ').strip()
if not re.fullmatch(r'[A-Za-z0-9_-]{1,60}', RUN_ID):
    raise ValueError('Invalid run ID')
RESUME = input('Type RESUME for an existing interrupted run; NEW for a new run: ').strip()
if RESUME not in ('RESUME', 'NEW'):
    raise ValueError('Choose RESUME or NEW explicitly')
RUN = PROJECT / 'experiments' / 'phase4-tversky' / RUN_ID
PREFLIGHT = PROJECT / 'experiments' / 'phase4-tversky' / 'preflight' / RUN_ID / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PREFLIGHT.mkdir(parents=True, exist_ok=False)
config = json.loads((REPO / 'configs/tversky.json').read_text())
seed_everything(config['seed'])
environment_text = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
environment = {'python': sys.version, 'torch': str(torch.__version__),
    'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
    'gpu': torch.cuda.get_device_name(0), 'packages': environment_text,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'cublas_workspace_config': os.environ['CUBLAS_WORKSPACE_CONFIG']}
write_json(PREFLIGHT/'environment.json', environment)
(PREFLIGHT/'environment.txt').write_text(environment_text)
print('Tversky run:', RUN)
print(json.dumps(config, indent=2))

assert config['precision'] == 'float32' and config['batch_size'] == 16
assert config['loss'] == {'name': 'TverskyLoss', 'alpha': .3, 'beta': .7, 'smooth': 1e-6, 'reduction': 'mean'}
bce_config = json.loads((REPO/'configs/bce.json').read_text())
assert {k for k in config if config[k] != bce_config[k]} == {'experiment', 'loss'}
BCE_RUN = PROJECT/'experiments'/'phase3-bce'/'bce-seed42-01'
bce_identity_path = BCE_RUN/'identity.json'
if not bce_identity_path.is_file():
    raise RuntimeError('Original BCE identity.json is required for a matched experiment; return this error')
bce_identity = json.loads(bce_identity_path.read_text())
assert bce_identity['config'] == bce_config, 'BCE configuration differs from the frozen source'
core_runtime = ('python', 'torch', 'cuda', 'cudnn', 'gpu', 'deterministic_algorithms', 'cublas_workspace_config')
runtime_differences = {k: [bce_identity['environment'].get(k), environment[k]]
    for k in core_runtime if bce_identity['environment'].get(k) != environment[k]}
if runtime_differences:
    raise RuntimeError(f'Core BCE/Tversky runtime differs; stop for review: {runtime_differences}')
bce_packages = set(bce_identity['environment']['packages'].splitlines())
new_packages = set(environment_text.splitlines())
package_differences = {'removed_or_changed': sorted(bce_packages-new_packages),
                       'added_or_changed': sorted(new_packages-bce_packages)}
write_json(PREFLIGHT/'package_differences_from_bce.json', package_differences)
(PREFLIGHT/'pytest.txt').write_text(test_run.stdout)
print('Supporting-package differences to review:', json.dumps(package_differences, indent=2))
def frozen_versions(lines):
    return {name.lower(): version for line in lines if '==' in line
            for name, version in [line.split('==', 1)]}
bce_versions, current_versions = frozen_versions(bce_packages), frozen_versions(new_packages)
for package in ('numpy', 'pillow', 'tifffile', 'imagecodecs'):
    if not bce_versions.get(package) or bce_versions.get(package) != current_versions.get(package):
        raise RuntimeError(f'Preprocessing dependency differs from BCE: {package}; stop for review')


## 6. Verify the original manifest and training-derived size cutoffs
Use the independent canonical identity supplied in the Phase 4 request, not a
newly generated hash. Verify original counts; no assignment or cutoff is recomputed.
Only aggregate counts are displayed. Phase 2 and Phase 3 remain read-only.


In [ ]:
from src.dataset import load_manifest, verify_dataset, MRIDataset, make_loader, file_sha256
EXPECTED_MANIFEST_HASH = '7514044281736b609e9d1c09c342a3f2514d0a2ca4dea9c4ce0d466cd86fe493'
manifest_path = PHASE2 / 'split_manifest.json'
manifest = load_manifest(manifest_path, EXPECTED_MANIFEST_HASH)
manifest_file_sha256 = file_sha256(manifest_path)
expected_counts = {'train': (77, 2719, 934), 'validation': (16, 621, 209), 'test': (17, 589, 230)}
for subset, expected in expected_counts.items():
    observed = manifest['summary'][subset]
    if tuple(observed[k] for k in ('patients', 'slices', 'positive_slices')) != expected:
        raise ValueError('Frozen split differs from the supplied verified Phase 2 counts')
print('Manifest identity:', manifest['manifest_sha256'])
print('Dataset fingerprint:', manifest['dataset_fingerprint'])
print('Original training size cutoffs:', manifest['size_cutoffs'])
print('Frozen counts:', json.dumps(manifest['summary'], indent=2))

assert bce_identity['manifest_sha256'] == EXPECTED_MANIFEST_HASH
assert bce_identity['size_cutoffs'] == manifest['size_cutoffs']


## 7. Access version-2 data in Colab; verify the canonical root
Read the [Kaggle release](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation),
[CC BY-NC-SA 4.0 terms](https://creativecommons.org/licenses/by-nc-sa/4.0/) and
upstream terms. Cached version-2 data is reused when available. If login is needed,
grant the Colab KAGGLE_API_TOKEN secret access or run kagglehub.login() with its
hidden prompt. Never send credentials back. No Windows dataset download.

Reuse the Phase 2 root resolver, including complete duplicate-directory hash
comparison. Verify every saved pair's bytes against the original manifest. This is
an identity preflight, not a new Phase 2 audit, new split or new size calibration.
Test TIFFs are checked for integrity only; only development pairs are decoded.


In [ ]:
from src.acquisition import download_dataset, resolve_dataset_root, get_metadata
release = get_metadata()
if release['metadata']['licenseName'] != 'CC BY-NC-SA 4.0' or release['metadata']['currentVersionNumber'] != 2:
    raise RuntimeError('Release metadata changed; stop for review')
write_json(PREFLIGHT/'dataset_release.json', release)
terms_accepted = input('After reviewing the dataset/upstream terms, type ACCEPT: ') == 'ACCEPT'
DOWNLOAD_ROOT = download_dataset(terms_accepted=terms_accepted)
DATA_ROOT, root_report = resolve_dataset_root(DOWNLOAD_ROOT)
verify_dataset(DATA_ROOT, manifest)
write_json(PREFLIGHT/'root_resolution.json', root_report)
print('Canonical root verified; no assignments regenerated or modified.')


## 8. Verify development loaders, unchanged U-Net and Tversky backward
Joint augmentation is training-only; validation is deterministic. No test Dataset.
Use the same seed and initialisation as BCE, never its trained checkpoint.


In [ ]:
from src.model import UNet
from src.training import evaluate, fit
from src.losses import make_criterion
train_data = MRIDataset(DATA_ROOT, manifest, 'train', **config['preprocessing'], seed=config['seed'])
val_data = MRIDataset(DATA_ROOT, manifest, 'validation', **config['preprocessing'], seed=config['seed'])
train_patients = {r['patient'] for r in train_data.records}
val_patients = {r['patient'] for r in val_data.records}
assert not train_patients & val_patients
batch = next(iter(make_loader(train_data, config)))
assert batch['image'].shape[1:] == (3, 256, 256)
assert batch['mask'].shape[1:] == (1, 256, 256)
assert set(batch['mask'].unique().tolist()) <= {0, 1}
device = torch.device('cuda')
seed_everything(config['seed'])
check_model = UNet(**config['model']).to(device)
with torch.no_grad():
    assert check_model(batch['image'][:2].to(device)).shape == (2, 1, 256, 256)
parameter_count = sum(p.numel() for p in check_model.parameters())
print('Parameters:', parameter_count, 'train/validation slices:', len(train_data), len(val_data))
assert parameter_count == 1942577
loss_check = make_criterion(config)(check_model(batch['image'][:2].to(device)), batch['mask'][:2].to(device))
loss_check.backward()
assert torch.isfinite(loss_check)
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in check_model.parameters())
print('Tversky real training-batch loss/backward:', loss_check.item())
del check_model, batch, loss_check
torch.cuda.empty_cache()


## 9. Inspect the frozen loss on synthetic edge cases
These are numerical checks, not MRI validation. A background-prediction positive
mask must push positive logits up; false positives must push background logits down.
At native resolution an empty mask with p=.5 may have loss rounded to 1 and a tiny
but finite FP gradient. This is a known limitation of the frozen convention.
Do not add BCE/hybrids or change smoothing after viewing validation results.


In [ ]:
loss_diagnostics = {}
for case in ('empty', 'tiny', 'large', 'background_prediction', 'nearly_perfect', 'mixed'):
    target = torch.zeros(2, 1, 256, 256, device=device)
    if case != 'empty':
        target[0, 0, 0, 0] = 1
    if case in ('large', 'background_prediction', 'nearly_perfect', 'mixed'):
        target[0, 0, :200, :200] = 1
    logits = torch.zeros_like(target)
    if case == 'background_prediction':
        logits.fill_(-12)
    if case == 'nearly_perfect':
        logits = torch.where(target.bool(), 12., -12.)
    logits.requires_grad_()
    value = make_criterion(config)(logits, target)
    value.backward()
    assert torch.isfinite(value) and torch.isfinite(logits.grad).all()
    assert (logits.grad[target == 0] > 0).all()
    if target.any():
        assert (logits.grad[target == 1] < 0).all()
    loss_diagnostics[case] = {'loss': value.item(), 'max_abs_gradient': logits.grad.abs().max().item(),
                              'min_abs_gradient': logits.grad.abs().min().item()}
write_json(PREFLIGHT/'loss_diagnostics.json', loss_diagnostics)
print(json.dumps(loss_diagnostics, indent=2))
del target, logits, value


## 10. Training-only smoke test and review gate
Run 100 updates on four fixed augmented positive training slices using Tversky.
Inspect loss decrease, tiny training Dice, checkpoint reload and FLAIR overlays.
These are debugging results, never validation evidence. Review package differences
and empty-mask gradient limitation too. Set SMOKE_REVIEWED=True only after review.
The definitive fit constructs a fresh seed42 model and optimiser; no smoke weights
or RNG state carry into it. If foreground fitting fails, stop and return evidence.


In [ ]:
from src.smoke import tiny_overfit
smoke = tiny_overfit(train_data, config, PREFLIGHT/'smoke', steps=100)
write_json(PREFLIGHT/'smoke.json', smoke)
print(json.dumps(smoke, indent=2))
from IPython.display import display, Image
display(Image(filename=str(PREFLIGHT/'smoke'/'smoke_debug.png')))


In [ ]:
SMOKE_REVIEWED = False  # USER INPUT: set True after reviewing the outputs above, then run this small cell.


## 11. Freeze provenance and train/resume Tversky
Type TRAIN after reviewing the smoke and loss convention. The only intervention
is loss. Adam .001, batch16, float32, max30, patience7, min_delta=.0001;
best validation positive-slice patient-macro Dice at .5. OOM stops; do not change
batch size now that BCE is complete. No schedule, weighting, AMP or postprocessing.
Resume requires the same config/commit/manifest/environment. Preserve all run files;
only an unfinished epoch is replayed, within the original budget.


In [ ]:
if not SMOKE_REVIEWED or not smoke['loss_decreased']:
    raise RuntimeError('Review the smoke outputs and set SMOKE_REVIEWED=True in section 10')
if input('Type TRAIN to run the frozen Tversky experiment: ') != 'TRAIN':
    raise RuntimeError('Training not confirmed')
identity = {'config': config, 'commit': REVISION, 'environment': environment,
    'manifest_sha256': manifest['manifest_sha256'], 'manifest_file_sha256': manifest_file_sha256,
    'dataset_fingerprint': manifest['dataset_fingerprint'], 'size_cutoffs': manifest['size_cutoffs'],
    'bce_reference_commit': bce_identity['commit'],
    'bce_identity_file_sha256': file_sha256(bce_identity_path),
    'parameter_count': parameter_count, 'channel_mapping': ['pre-contrast', 'FLAIR', 'post-contrast']}
write_json(PREFLIGHT/'frozen_protocol.json', identity)
try:
    model, history = fit(train_data, val_data, config, RUN, identity, resume=RESUME == 'RESUME')
except BaseException as error:
    write_json(PREFLIGHT/'training_failure.json', {'type': type(error).__name__,
        'message': str(error), 'commit': REVISION, 'run': str(RUN),
        'manifest_sha256': manifest['manifest_sha256'], 'test_evaluated': False})
    raise
print('Completed epochs:', len(history))


## 12. Validation, frozen threshold selection and BCE comparison
Evaluate the selected best checkpoint at .5 and the declared .10:.05:.90 search.
Maximise positive-slice patient-macro Dice subject to empty-mask FP <=10%; ties:
lower FP then higher threshold. Fallback .5 with constraint-failure status.
Use original size cutoffs. Reports include IoU, denominators and pooled metrics.
Compare with supplied rounded BCE aggregates; missing BCE metrics stay unavailable.
Differences are **percentage points**, Tversky minus BCE. Review small-region
recall/Dice, precision and FP trade-offs; no improvement claim from recall alone.
Numerical BCE and Tversky training losses are different objectives.


In [ ]:
from src.metrics import select_threshold
from src.reporting import save_curves, save_predictions, compare_validation
validation = evaluate(model, make_loader(val_data, config), device, manifest['size_cutoffs'],
                      config['thresholds'], make_criterion(config))
selection = select_threshold(validation['reports'], config['empty_fp_limit'])
selected = selection['threshold']
reports = {'default_threshold': .5, 'default': validation['reports'][.5],
           'selection': selection, 'selected': validation['reports'][selected],
           'threshold_search': validation['reports'], 'validation_loss': validation['loss']}
bce_reference = json.loads((REPO/'docs/bce-validation-reference.json').read_text())
comparison = compare_validation(reports, bce_reference, fp_limit=config['empty_fp_limit'],
                                fp_increase_limit=config['comparison_fp_increase_limit'])
write_json(RUN/'validation_reports.json', reports)
write_json(RUN/'bce_validation_reference.json', bce_reference)
write_json(RUN/'validation_comparison.json', comparison)
save_curves(history, RUN, loss_name='Tversky')
save_predictions(model, val_data, device, .5, RUN/'figures'/'default')
save_predictions(model, val_data, device, selected, RUN/'figures'/'selected')
print(json.dumps({'selection': selection, 'comparison': comparison}, indent=2))
from IPython.display import display, Image
display(Image(filename=str(RUN/'training_curves.png')))
for path in sorted((RUN/'figures').rglob('*.png')):
    display(Image(filename=str(path)))


## 13. Persist private artifacts and verify read-only evidence
Keep the entire run directory, including two rotating state checkpoints and RNG/
optimiser state. Only load your own trusted checkpoints. Original manifest and
BCE identity file must retain their byte hashes. No Phase 2/3 output is written.


In [ ]:
assert file_sha256(manifest_path) == manifest_file_sha256, 'Frozen manifest changed: STOP'
assert file_sha256(bce_identity_path) == identity['bce_identity_file_sha256'], 'BCE identity changed: STOP'
(RUN/'environment.txt').write_text(environment_text)
(RUN/'pytest.txt').write_text(test_run.stdout)
write_json(RUN/'environment.json', environment)
write_json(RUN/'config.json', config)
write_json(RUN/'smoke_debug_only.json', smoke)
write_json(RUN/'loss_diagnostics.json', loss_diagnostics)
write_json(RUN/'package_differences_from_bce.json', package_differences)
best_epoch = max(history, key=lambda r: r['validation']['positive_patient_macro']['dice'])['epoch']
summary = {'stage': 'Phase 4 Tversky validation only', 'commit': REVISION,
    'manifest_sha256': manifest['manifest_sha256'], 'parameter_count': parameter_count,
    'completed_epochs': len(history), 'best_epoch': best_epoch,
    'size_cutoffs': manifest['size_cutoffs'], 'loss_convention': config['loss'],
    'tversky_default': reports['default'], 'selection': selection,
    'tversky_selected': reports['selected'], 'bce_comparison': comparison,
    'smoke_debug_only': smoke, 'test_evaluated': False, 'phase5_started': False,
    'training_seconds': sum(r['seconds'] for r in history)}
write_json(RUN/'review_summary.json', summary)
write_json(RUN/'artifact_hashes.json', {p.relative_to(RUN).as_posix(): file_sha256(p)
    for p in sorted(RUN.rglob('*')) if p.is_file() and p.name != 'artifact_hashes.json'})
print('Persistent private run:', RUN)
print('Best checkpoint SHA256:', file_sha256(RUN/'best.pt'))


## 14. Compact review and hand back; stop before Phase 5
Return review_summary.json, pytest output, smoke output/overlay, training curves,
and representative validation overlays at both thresholds. Report failures and
negative findings honestly. Do not send patient records, tokens, raw MRI or weights.

After reset: same SHA, sections 1–10, same run ID with RESUME, review smoke, 11–14.
No Phase 5 until actual Tversky results have been reviewed and scope authorised.
Sources: [Tversky paper](https://arxiv.org/abs/1706.05721),
[U-Net](https://arxiv.org/abs/1505.04597),
[PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html).
Prepared with OpenAI Codex assistance; the student reviews, runs and interprets.


In [ ]:
print('Epochs:', summary['completed_epochs'], 'Best epoch:', summary['best_epoch'])
print('Threshold:', selected, 'Status:', selection['status'])
print('Point | Metric | BCE | Tversky | Difference (pp)')
for point, values in comparison['operating_points'].items():
    for metric in ('dice', 'recall', 'precision', 'small_dice', 'small_recall', 'empty_fp_rate'):
        value, delta = values['tversky'][metric], values['difference_pp'][metric]
        shown = 'NA' if value is None else f'{value:.5f}'
        change = 'NA' if delta is None else f'{delta:+.2f}'
        print(f"{point} | {metric} | {values['bce'][metric]:.5f} | {shown} | {change}")
print('Full private summary:')
print((RUN/'review_summary.json').read_text())
